In [ ]:
import marimo as mo
import pandas as pd

In [ ]:
raw_df = pd.read_csv("course_lead_scoring_2026.csv")
target = "converted"
categorical_features = [
    "lead_source",
    "industry",
    "employment_status",
    "location",
]
numerical_features = [
    "annual_income",
    "number_of_courses_viewed",
    "interaction_count",
    "lead_score",
]
raw_df

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
...,...,...,...,...,...,...,...,...,...
4995,organic_search,education,employed,asia,45834.0,2,2,0.37,1
4996,NaN,technology,employed,north_america,84831.0,2,5,0.51,0
4997,organic_search,finance,employed,asia,53636.0,4,6,0.75,1
4998,referral,technology,employed,africa,69430.0,4,8,0.79,1


## Data preparation

In [ ]:
missing_values = raw_df.isna().sum()
missing_values

,0
lead_source,148
industry,240
employment_status,194
location,208
annual_income,369
number_of_courses_viewed,0
interaction_count,0
lead_score,35
converted,0


In [ ]:
df = raw_df.copy()
df[categorical_features] = df[categorical_features].fillna("NA")
df[numerical_features] = df[numerical_features].fillna(0.0)
df.isna().sum()

,0
lead_source,0
industry,0
employment_status,0
location,0
annual_income,0
number_of_courses_viewed,0
interaction_count,0
lead_score,0
converted,0


## Question 1 - What is the mode for `industry`?

In [ ]:
industry_mode = df["industry"].mode().iloc[0]
industry_mode

'technology'

## Question 2 - Which numerical features have the biggest correlation?

In [ ]:
correlation_matrix = df[numerical_features].corr()
correlation_matrix

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [ ]:
correlation_options = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]
correlation_scores = {
    pair: correlation_matrix.loc[pair[0], pair[1]]
    for pair in correlation_options
}
most_correlated_features = max(
    correlation_scores, key=correlation_scores.get
)
correlation_scores, most_correlated_features

## Split the data

In [ ]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42
)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_train = df_train[target]
y_val = df_val[target]
X_train = df_train.drop(columns=target)
X_val = df_val.drop(columns=target)
X_train, X_val, y_train, y_val

## Question 3 - Which categorical feature has the biggest mutual information?

In [ ]:
from sklearn.metrics import mutual_info_score

mutual_information_scores = {
    feature: round(mutual_info_score(y_train, X_train[feature]), 2)
    for feature in categorical_features
}
most_informative_feature = max(
    mutual_information_scores, key=mutual_information_scores.get
)
mutual_information_scores, most_informative_feature

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def evaluate_logistic_regression(
    X_train,
    y_train,
    X_eval,
    y_eval,
    categorical_features,
    C=1.0,
):
    X_train_encoded = pd.get_dummies(
        X_train, columns=categorical_features, dtype=int
    )
    X_eval_encoded = pd.get_dummies(
        X_eval, columns=categorical_features, dtype=int
    ).reindex(columns=X_train_encoded.columns, fill_value=0)

    model = LogisticRegression(
        solver="liblinear", C=C, max_iter=1000, random_state=42
    )
    model.fit(X_train_encoded, y_train)
    accuracy = accuracy_score(y_eval, model.predict(X_eval_encoded))
    return accuracy

## Question 4 - What is the validation accuracy?

In [ ]:
accuracy = evaluate_logistic_regression(
    X_train,
    y_train,
    X_val,
    y_val,
    categorical_features,
)
rounded_accuracy = round(accuracy, 2)
rounded_accuracy

0.65

## Question 5 - Which feature has the smallest accuracy difference when removed?

In [ ]:
features_to_eliminate = [
    "lead_source",
    "number_of_courses_viewed",
    "interaction_count",
]
accuracy_differences = {}
for feature in features_to_eliminate:
    remaining_categorical_features = [
        column
        for column in categorical_features
        if column != feature
    ]
    accuracy_without_feature = evaluate_logistic_regression(
        X_train.drop(columns=feature),
        y_train,
        X_val.drop(columns=feature),
        y_val,
        remaining_categorical_features,
    )
    accuracy_differences[feature] = accuracy - accuracy_without_feature

least_useful_feature = min(
    accuracy_differences, key=accuracy_differences.get
)
accuracy_differences, least_useful_feature

## Question 6 - Which regularization value gives the best accuracy?

In [ ]:
regularization_values = [0.000001, 0.00001, 0.0001, 0.001]
accuracy_by_C = {
    C: round(
        evaluate_logistic_regression(
            X_train,
            y_train,
            X_val,
            y_val,
            categorical_features,
            C=C,
        ),
        3,
    )
    for C in regularization_values
}
best_C = max(regularization_values, key=lambda C: (accuracy_by_C[C], -C))
accuracy_by_C, best_C

In [ ]:
mo.md(f"""
**Answers**

1. `industry` mode: **{industry_mode}**
2. Most correlated pair: **{most_correlated_features[0]}** and **{most_correlated_features[1]}**
3. Highest mutual information: **{most_informative_feature}**
4. Validation accuracy: **{rounded_accuracy}**
5. Smallest accuracy difference: **{least_useful_feature}** (`{accuracy_differences[least_useful_feature]:.3f}`)
6. Best regularization parameter: **{best_C}**
""")

**Answers**

1. `industry` mode: **technology**
2. Most correlated pair: **interaction_count** and **lead_score**
3. Highest mutual information: **lead_source**
4. Validation accuracy: **0.65**
5. Smallest accuracy difference: **number_of_courses_viewed** (`0.002`)
6. Best regularization parameter: **0.001**